# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [44]:
from datetime import datetime
from agent import Agent
from ragas import evaluate
from datasets import Dataset
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.metrics import AspectCritic

from langchain_core.messages import SystemMessage, HumanMessage


import json
from dotenv import load_dotenv


/var/folders/qm/3pxfpqw17hl7x_xzj3nv79bw0000gn/T/ipykernel_50809/1420012161.py:7: DeprecationWarning: Importing AspectCritic from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AspectCritic
  from ragas.metrics import AspectCritic


In [21]:
load_dotenv()

# Embeddings
embeddings = OpenAIEmbeddings(
    base_url="https://openai.vocareum.com/v1",
    api_key=os.getenv("VOCAREUM_API_KEY")
)


In [3]:
## TODO: Create the agent's instructions

ECOHOME_SYSTEM_PROMPT = """
You are EcoHome, a residential energy optimization assistant.

ROLE:
Help users reduce electricity costs and optimize energy usage using weather forecasts and time-of-use electricity pricing data.

WHAT YOU SHOULD DO:
1. Understand the user's goal (EV charging, HVAC, battery use, load shifting, cost savings).
2. Use tools (get_weather_forecast, get_electricity_prices) when data is needed.
3. Analyze peak vs off-peak pricing, demand charges, temperature impact, and solar irradiance.
4. Provide structured, actionable recommendations.

RECOMMENDATION FORMAT (always include):
- Summary of Findings (key pricing/weather insights)
- Recommended Actions (specific hours + actions)
- Estimated Cost Impact (if possible)
- Assumptions (clearly state any)

KEY CAPABILITIES:
- Load shifting optimization
- EV charging scheduling
- Battery charge/discharge strategy
- HVAC timing recommendations
- Solar alignment with pricing
- Demand charge avoidance

STYLE:
Clear, structured, concise, and data-driven.

EXAMPLE QUESTIONS:
- “When should I charge my EV?”
- “What's the cheapest time to run appliances?”
- “How do I avoid peak pricing today?”
- “When should I use my battery?”
"""



In [4]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [5]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [6]:
print(response["messages"][-1].content)

### Summary of Findings
- **Weather Forecast**: Tomorrow in San Francisco, solar irradiance is expected to peak between 12 PM and 2 PM, with the highest value at 1 PM (995.37 W/m²). The day will start with cloudy conditions, transitioning to sunny around noon.
- **Electricity Pricing**: 
  - **Peak Pricing**: 6 AM to 7 PM with rates ranging from $0.155 to $0.173 per kWh.
  - **Off-Peak Pricing**: 12 AM to 6 AM and 10 PM to 11:59 PM with rates as low as $0.084 per kWh.

### Recommended Actions
1. **Charge your EV during off-peak hours**:
   - **Best Time**: Start charging at **10 PM** tonight and continue until **6 AM** tomorrow morning.
   - **Alternative Time**: If you prefer to charge during the day, consider charging from **2 PM to 4 PM** when solar power is abundant, but be aware that the rates will be higher during this time.

### Estimated Cost Impact
- **Charging Overnight (10 PM - 6 AM)**:
  - Duration: 8 hours
  - Average Rate: $0.086 (approx.)
  - Estimated Cost for 10 kWh: $

In [7]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

TOOLS:
- get_weather_forecast
- get_electricity_prices


## 2. Define Test Cases

In [8]:
# TODO: Define comprehensive test cases for the Energy Advisor
# Create 10 test cases covering different scenarios:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations

In [9]:
test_cases = [
    {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "Recommends specific charging window(s) aligned to lowest rates and highest solar irradiance; explains tradeoffs and provides clear next steps."
    },
    {
        "id": "ev_charging_2",
        "question": "My EV needs 30 kWh by 7am. What's the cheapest charging schedule tonight?",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "Uses hourly prices to pick cheapest hours; checks feasibility to deliver 30 kWh by 7am; estimates cost vs a peak-time alternative and shows savings."
    },
    {
        "id": "thermostat_1",
        "question": "It's getting cold this week. What thermostat schedule should I use to save money without losing comfort?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "search_energy_tips"],
        "expected_response": "Proposes a weekday/weekend schedule (sleep/away/home) tied to forecast and peak/off-peak; includes comfort notes and practical tips (e.g., setbacks, fan, sealing)."
    },
    {
        "id": "thermostat_2",
        "question": "Why did my HVAC usage spike over the last 7 days compared to the week before?",
        "expected_tools": ["query_energy_usage", "get_weather_forecast"],
        "expected_response": "Compares two date ranges for HVAC usage; relates changes to weather (temperature/conditions); suggests likely causes and targeted actions."
    },
    {
        "id": "appliance_scheduling_1",
        "question": "When should I run my dishwasher and laundry each day to minimize my bill?",
        "expected_tools": ["get_electricity_prices"],
        "expected_response": "Identifies off-peak hours and recommends appliance run windows; includes simple scheduling plan and mentions avoiding peak periods."
    },
    {
        "id": "appliance_scheduling_2",
        "question": "I work from home and cook a lot. What are the top three things I can shift to off-peak hours?",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "Recommends 3 shiftable loads (e.g., laundry, dishwasher, pre-cooling/heating, EV) with time windows from TOU rates and practical best practices."
    },
    {
        "id": "solar_max_1",
        "question": "How can I maximize the use of my solar power tomorrow instead of pulling from the grid?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "Uses forecast solar irradiance and price periods to suggest when to run loads; recommends midday usage, pre-conditioning, and explains grid vs solar tradeoffs."
    },
    {
        "id": "solar_max_2",
        "question": "My solar production seems lower this month. Can you check what changed?",
        "expected_tools": ["query_solar_generation", "get_weather_forecast"],
        "expected_response": "Pulls solar generation data over an appropriate range; compares to expected conditions/irradiance; suggests causes (weather, shading, soiling) and next steps."
    },
    {
        "id": "cost_savings_1",
        "question": "If I reduce my HVAC usage from 22 kWh/day to 16 kWh/day, how much would I save?",
        "expected_tools": ["calculate_energy_savings", "get_electricity_prices"],
        "expected_response": "Calculates daily and annual savings using price per kWh (prefer from tool); reports kWh and $ savings and percent reduction; notes assumptions."
    },
    {
        "id": "cost_savings_2",
        "question": "Based on my last 30 days, what device type is costing me the most and what's a realistic savings target?",
        "expected_tools": ["query_energy_usage", "search_energy_tips", "calculate_energy_savings"],
        "expected_response": "Analyzes 30-day usage by device_type; identifies top cost driver; proposes a realistic optimized target and quantifies potential savings with clear actions."
    },
]


if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

## 3. Run Agent Tests

In [10]:
CONTEXT = "Location: San Francisco, CA"

In [11]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: ev_charging_2
Question: My EV needs 30 kWh by 7am. What's the cheapest charging schedule tonight?
--------------------------------------------------

Test 3: thermostat_1
Question: It's getting cold this week. What thermostat schedule should I use to save money without losing comfort?
--------------------------------------------------

Test 4: thermostat_2
Question: Why did my HVAC usage spike over the last 7 days compared to the week before?
--------------------------------------------------

Test 5: appliance_scheduling_1
Question: When should I run my dishwasher and laundry each day to minimize my bill?
--------------------------------------------------

Test 6: appliance_scheduling_2
Question: I work from home and cook a lot. What are the top three things I can shift to off-p

In [12]:
test_results

[{'test_id': 'ev_charging_1',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'response': {'messages': [SystemMessage(content='Location: San Francisco, CA', additional_kwargs={}, response_metadata={}, id='03c1563d-18f1-4656-b1bc-e77223c85cb4'),
    HumanMessage(content='When should I charge my electric car tomorrow to minimize cost and maximize solar power?', additional_kwargs={}, response_metadata={}, id='d0fc1c21-3973-40e4-a306-87af73604ca0'),
    AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 1111, 'total_tokens': 1172, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1024}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'id': 

## 4. Evaluate Responses

In [13]:
# TODO: Implement evaluation functions
# Create functions to evaluate:
# - Final Response
# - Tool usage

In [23]:
## Instantiate LLM Judge

llm_judge = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    base_url="https://openai.vocareum.com/v1",
    api_key=os.getenv("VOCAREUM_API_KEY"),
)


In [32]:
# TODO: Create a response evaluator
def _to_text(x):
    # LangChain message object
    if hasattr(x, "content"):
        return x.content
    # List of messages -> join
    if isinstance(x, list):
        parts = []
        for m in x:
            if hasattr(m, "content"):
                parts.append(m.content)
            else:
                parts.append(str(m))
        return "\n".join(parts)
    # Fallback
    return str(x)

def extract_retrieved_contexts(response):
    """
    Extract tool outputs from a LangChain agent response
    and return them as list of strings (for RAGAS).
    """
    messages = response["messages"] if "messages" in response else response
    
    retrieved_contexts = []

    for msg in messages:
        # ToolMessage objects have .type == "tool"
        if getattr(msg, "type", None) == "tool":
            try:
                # Pretty-print JSON if possible
                content = json.loads(msg.content)
                retrieved_contexts.append(json.dumps(content, indent=2))
            except Exception:
                retrieved_contexts.append(str(msg.content))

    return retrieved_contexts

def evaluate_response(question, final_response, expected_response):
    """Evaluate a single response against expected response"""

    retrieved_contexts = extract_retrieved_contexts(final_response)

    dataset = Dataset.from_dict(
        {
            "question": [question],
            "answer": [_to_text(final_response)],
            "contexts": [retrieved_contexts],
            "ground_truth": [expected_response]
        }
    )

    accuracy = AspectCritic(
    name="accuracy",
    definition="Are the claims and recommendations correct given the provided context/tools? Flag any incorrect statements."
    )

    relevance = AspectCritic(
        name="relevance",
        definition="Does the answer directly address the user question and avoid irrelevant content?"
    )

    completeness = AspectCritic(
        name="completeness",
        definition="Does the answer cover all key parts of the question (cost + solar + actionable timing + tradeoffs)?"
    )

    usefulness = AspectCritic(
        name="usefulness",
        definition="Is the answer actionable with clear steps, specific time windows, and practical advice?"
    )

    evaluation_results = evaluate(
        dataset=dataset,
        metrics=[accuracy, relevance, completeness, usefulness],
        llm=llm_judge,
        embeddings=embeddings
    )
    print(evaluation_results)
    
    return evaluation_results

In [37]:
# TODO: Create a tool udage evaluator
def evaluate_tool_usage(messages, expected_tools):
    """Evaluate if the right tools were used"""
    expected_set = set(expected_tools or [])
    used_set = set()

    # Extract tool calls from AI messages
    for msg in messages:
        tool_calls = getattr(msg, "tool_calls", None)
        if tool_calls:
            for tc in tool_calls:
                if isinstance(tc, dict):
                    name = tc.get("name")
                else:
                    name = getattr(tc, "name", None)
                if name:
                    used_set.add(name)

    # ---- Metrics ----
    # Completeness = recall
    tool_completeness = (
        len(expected_set & used_set) / len(expected_set)
        if expected_set else 1.0
    )

    # Appropriateness = precision
    tool_appropriateness = (
        len(expected_set & used_set) / len(used_set)
        if used_set else (1.0 if not expected_set else 0.0)
    )

    return {
        "tool_completeness": round(tool_completeness, 4),
        "tool_appropriateness": round(tool_appropriateness, 4),
    }


In [17]:
# TODO: Generate a comprehensive evaluation report
# Calculate overall scores and metrics
# Identify strengths and weaknesses
# Provide recommendations for improvement
def generate_evaluation_report():
    pass

In [46]:
# # Tests
sample_test = test_results[0]
# print(sample_test)

response_evals = [evaluate_response(test["question"], test["response"], test["expected_response"] ) for test in test_results]
tool_evals = [evaluate_tool_usage(test["response"]["messages"], test["expected_tools"]) for test in test_results ]


# evaluate_response(sample_test["question"], sample_test["response"], sample_test["expected_response"])

# tools_accuracy = evaluate_tool_usage(sample_test["response"]["messages"], sample_test["expected_tools"])

# print(tools_accuracy)



Evaluating: 100%|██████████| 4/4 [00:03<00:00,  1.20it/s]


{'accuracy': 1.0000, 'relevance': 1.0000, 'completeness': 1.0000, 'usefulness': 1.0000}


Evaluating: 100%|██████████| 4/4 [00:05<00:00,  1.26s/it]


{'accuracy': 1.0000, 'relevance': 1.0000, 'completeness': 1.0000, 'usefulness': 1.0000}


Evaluating: 100%|██████████| 4/4 [00:07<00:00,  1.81s/it]


...

{'accuracy': 1.0000, 'relevance': 1.0000, 'completeness': 0.0000, 'usefulness': 1.0000}


Evaluating: 100%|██████████| 4/4 [00:02<00:00,  1.46it/s]


{'accuracy': 1.0000, 'relevance': 1.0000, 'completeness': 1.0000, 'usefulness': 1.0000}


Evaluating: 100%|██████████| 4/4 [00:03<00:00,  1.31it/s]


{'accuracy': 0.0000, 'relevance': 0.0000, 'completeness': 0.0000, 'usefulness': 1.0000}


In [43]:
# This will build the input payload to the llm to generate a comprehensive report
def build_report_payload(test_results, response_evals, tool_evals):
    payload = []
    n = min(len(test_results), len(response_evals), len(tool_evals))

    for i in range(n):
        t = test_results[i]
        r = response_evals[i]
        u = tool_evals[i]

        payload.append({
            "test_id": t.get("test_id") or t.get("id"),
            "question": t.get("question"),
            "expected_tools": t.get("expected_tools"),
            "tool_metrics": {
                "tool_appropriateness": u.get("tool_appropriateness"),
                "tool_completeness": u.get("tool_completeness"),
            },
            "response_metrics": r,  # already a dict of ragas scores
            "answer_excerpt": (t.get("response", {})
                                .get("messages", [])[-1].content[:500]
                              ) if t.get("response") else None,
        })

    return payload

In [47]:
REPORT_SYSTEM = """You are an evaluation analyst.
You will be given per-test evaluation results for an Energy Advisor agent:
- response_metrics (Ragas scores)
- tool_metrics (tool_appropriateness, tool_completeness)
- expected_tools and question
- a short excerpt of the agent answer

Write a comprehensive evaluation report with:
1) Overall summary of performance
2) Strengths (bullet list)
3) Weaknesses (bullet list)
4) Concrete recommendations to improve the agent (bullet list)
5) Call out 2–3 best tests and 2–3 worst tests with reasons

Be specific and reference evidence from the metrics and excerpts.
Return STRICT JSON with keys:
summary, strengths, weaknesses, recommendations, best_cases, worst_cases.
"""

def generate_evaluation_report_llm(test_results, response_evals, tool_evals, llm_judge):
    payload = build_report_payload(test_results, response_evals, tool_evals)

    prompt = json.dumps({"tests": payload}, indent=2)

    msg = [
        SystemMessage(content=REPORT_SYSTEM),
        HumanMessage(content=prompt)
    ]

    out = llm_judge.invoke(msg).content

    # Try to parse JSON; if it fails, return raw text too
    try:
        return json.loads(out)
    except Exception:
        return {"raw_report": out, "note": "LLM did not return valid JSON."}

In [48]:
report = generate_evaluation_report_llm(test_results, response_evals, tool_evals, llm_judge)
print(report)

TypeError: Object of type EvaluationResult is not JSON serializable